In [ ]:
import tkinter as tk
from tkinter import ttk
import random
from tabulate import tabulate

canvas_width = 600
canvas_height = 400
lane_width = 80 
gap = 30 
LANE_NAMES = ["Smouha", "Mandara", "Sidi Gaber", "El Awayed"]

root = tk.Tk()
root.geometry("800x600")
root.title("Traffic Light Cycle Optimizer")

random.seed(42)

def Algorithm(Algorithm_name):
    random.seed(42) # random is same every time runned
    lanes = [0,0,0,0] # 0 cars fel awel
    waiting_time = [0,0,0,0] 
    total_arrivals = [0,0,0,0]
    max_queue = []
    total_cleared = 0
    cars_cleared_per_lane = [0, 0, 0, 0] 
    avg = [0,0,0,0]
        
    for cycle in range (20) : # 20 eshara
        for i in range (4) : # for each lane
            arrivals = random.randint(5,15) # mn 5 l 15
            lanes [i] += arrivals
            total_arrivals[i] += arrivals

        for i in range(4):
            waiting_time[i] += lanes[i]     
            
        before_algorithm = list(lanes) # take a copy to compare
        
        maximum = max(before_algorithm)
        max_queue.append(maximum)
        
        if Algorithm_name.lower() == "greedy" : # lowercase
            print(f" cycle {cycle+1}") 
            queue=[] # storing lane info
            for i in range(4) : # 4 to 4 lanes
                queue.append((lanes[i] , -i) ) # "-" if n of cars = i
                
            queue.sort() # smalles to largest
            car ,neg_index = queue.pop() 
            
            index= (-neg_index) # return to +
            print(f"greedy choose lane {index + 1} becouse it has {car} cars")
            cleared = min(car , 5)
            lanes[index] -= cleared # removing cars from lane choosed
            total_cleared += cleared
            cars_cleared_per_lane[index] += cleared
            
        elif Algorithm_name.lower() =="round-robin":
            print(f" cycle {cycle+1}") 
            lane_to_serve = cycle % 4 
            print(f"round-robin choose lane {lane_to_serve+1}  ")
            cleared  = min (lanes[lane_to_serve] , 5)  
            lanes[lane_to_serve] -= cleared
            total_cleared += cleared 
            cars_cleared_per_lane[lane_to_serve] += cleared
            
        else :
            print ("algorithm not found")
            return
        
        after_algorithm= list(lanes)
        print (f"before Algorithm {before_algorithm} after Algorithm {after_algorithm}")
        
    for i in range(4):
        avg[i] = waiting_time[i] / total_arrivals[i]
       
    return avg , total_cleared , max(max_queue)

def visual_updates(canvas, main_window, lanes, cycle_num, stats_label):
    if cycle_num == 1: 
        lanes = [0,0,0,0]
        random.seed(42)
    
    if cycle_num > 20: 
        stats_label.config(text="Simulation Finished!", fg="blue")
        return
    
    canvas.delete("all")

    for i in range(len(lanes)):
        lanes[i] += random.randint(5, 15)

    lane_to_serve = lanes.index(max(lanes))

    canva_h = 400
    lane_w = 70  
    gap = 30         
    start_point = 80
    max_visual_h = 250
    
    highest_car_count = max(lanes) if max(lanes) > 0 else 1
    scale = max_visual_h / max(highest_car_count, 50) 

    canvas.create_text(300, 20, text=f"Current Cycle: {cycle_num}", fill="black", font=("Arial", 16, "bold"))
    stats_label.config(text=f"Current Data (Lanes): {lanes}", fg="red")

    current_x = start_point
    for i in range(len(lanes)):
        num_cars = lanes[i]
        rect_height = num_cars * scale

        x1 = current_x
        y1 = canva_h - 70 
        x2 = current_x + lane_w
        y2 = y1 - rect_height

        if i == lane_to_serve:
            fill_color = "#2ecc71"   # green
            outline_color = "black"
            line_thickness = 2       
        else:
            fill_color = "#e74c3c"   # red
            outline_color = "black"
            line_thickness = 1

        canvas.create_rectangle(x1, y1, x2, y2, fill=fill_color, outline=outline_color, width=line_thickness)
        canvas.create_text(current_x + lane_w // 2, y1 + 30, text=LANE_NAMES[i], font=("Arial", 10, "bold"))
        canvas.create_text(current_x + lane_w // 2, y2 - 15, text=f"{num_cars}", font=("Arial", 10, "bold"))

        current_x += lane_w + gap

    cleared = min(lanes[lane_to_serve], 5)
    lanes[lane_to_serve] -= cleared

    main_window.after(750, lambda: visual_updates(canvas, main_window, lanes, cycle_num + 1, stats_label))
    
def show_comparison_table():
    greedy_avg, greedy_total_cleared, greedy_max = Algorithm("greedy")
    round_avg, round_total_cleared, round_max = Algorithm("round-robin")
    
    comparaison_table = [
        ["Greedy", *[f"{v:.4f}" for v in greedy_avg], greedy_total_cleared, greedy_max],
        ["Round-Robin", *[f"{v:.4f}" for v in round_avg], round_total_cleared, round_max]
    ]
    headers = ["Method", "Smouha", "Mandara", "Sidi Gaber", "El Awayed", "Total cleared", "Max queue"] 
    print(tabulate(comparaison_table, headers, tablefmt="grid"))

    table_win = tk.Toplevel(root)
    table_win.title("Comparison Table")
    table_win.geometry("950x350")
    
    cols = ("Method", "Smouha", "Mandara", "Sidi Gaber", "El Awayed", "Total cleared cars", "Max queue")
    tree = ttk.Treeview(table_win, columns=cols, show="headings")
    
    for col in cols:
        tree.heading(col, text=col)
        tree.column(col, width=110, anchor="center")
    
    greedy_row = ("Greedy", *[f"{val:.4f}" for val in greedy_avg], greedy_total_cleared, greedy_max)
    rr_row = ("Round-Robin", *[f"{val:.4f}" for val in round_avg], round_total_cleared, round_max) 
    
    tree.insert("", "end", values=greedy_row)
    tree.insert("", "end", values=rr_row)
    
    tree.pack(pady=20, padx=10, fill="both", expand=True)

label = tk.Label(root, text="Traffic Light Cycle", width=40, height=1, font=("Arial", 14, "bold"), fg="red")
label.pack(side=tk.TOP, pady=10)

canvas = tk.Canvas(
    root, 
    width=canvas_width, 
    height=canvas_height, 
    bg="#ecf0f1",  # white
    highlightthickness=1, 
    highlightbackground="#bdc3c7" # gray
)
canvas.pack(pady=20)

compare_button = tk.Button(
    root,
    text="Show Comparison Table", 
    width=25,
    bg="green",
    fg="white", 
    font=("Arial", 12, "bold"),
    command=show_comparison_table
)
compare_button.pack(padx=10)

test_lanes = [0,0,0,0]
mainbutton = tk.Button(
    root,
    text="Start Greedy Simulation", 
    width=20, 
    bg="purple", 
    fg="white", 
    font=("Arial", 12, "bold"),
    command=lambda: visual_updates(canvas, root, test_lanes, 1, label)
)
mainbutton.pack(side=tk.BOTTOM, pady=20)

root.mainloop()